# ⚡ Duino API — Google Colab

**Cloud-agnostic hyperscale AI inference — Gemma 4 · Streaming · React UI**

> 💡 **Before running:** `Runtime → Change runtime type → T4 GPU`
> 🔑 **HF Token:** Add `HF_TOKEN` to Colab Secrets (🔑 icon in sidebar)

---

## 📦 Cell 1 — Clone & Install
_Run once (~2 min). Fixes all known Colab Python 3.12 setuptools issues._

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/jalalmansour/Duino_API.git'
REPO_DIR = '/content/Duino_API'

# ── Clone / update ────────────────────────────────────────────────────────────
if not os.path.exists(REPO_DIR):
    print('Cloning Duino API...')
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    print('Repo found — pulling latest...')
    !git -C {REPO_DIR} pull --quiet

# ── Register on sys.path BEFORE any imports ───────────────────────────────────
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f'CWD: {os.getcwd()}')

# ── Step 1: Upgrade setuptools & wheel ────────────────────────────────────────
# CRITICAL: fixes 'BackendUnavailable: setuptools.backends.legacy' on Colab Py 3.12
print('Upgrading setuptools + wheel...')
!pip install --upgrade setuptools wheel pip -q

# ── Step 2: Core requirements ─────────────────────────────────────────────────
print('Installing core requirements...')
!pip install -r /content/Duino_API/requirements.txt -q

# ── Step 3: Install duino-api package ────────────────────────────────────────
# Try editable with --no-build-isolation first (avoids legacy backend lookup)
# Fall back to regular install if that also fails
print('Installing duino-api package...')
r = subprocess.run(
    ['pip', 'install', '-e', REPO_DIR, '-q', '--no-build-isolation'],
    capture_output=True, text=True
)
if r.returncode == 0:
    print('✅ duino-api installed (editable)')
else:
    print(f'  editable failed ({r.stderr[-200:].strip()}) — trying regular install...')
    r2 = subprocess.run(
        ['pip', 'install', REPO_DIR, '-q'],
        capture_output=True, text=True
    )
    if r2.returncode == 0:
        print('✅ duino-api installed (regular)')
    else:
        print(f'⚠️  Package install failed (sys.path fallback active):\n{r2.stderr[-300:]}')

print('\n✅ Setup complete — ready to start!')

## 🔑 Cell 2 — Set HuggingFace Token
_Required for Gemma 4. Use Colab Secrets (🔑 sidebar) — never paste tokens in code._

In [ ]:
import os

# ── Method A: Colab Secrets (RECOMMENDED) ─────────────────────────────────────
# Click the 🔑 icon in the left sidebar → add 'HF_TOKEN' and 'NGROK_AUTHTOKEN'
# They load automatically into os.environ.

# ── Method B: Temporary env var (not saved to notebook) ───────────────────────
# os.environ['HF_TOKEN']        = 'hf_...'   # from huggingface.co/settings/tokens
# os.environ['NGROK_AUTHTOKEN'] = '...'       # free at ngrok.com

# ── Status ────────────────────────────────────────────────────────────────────
hf    = bool(os.environ.get('HF_TOKEN'))
ngrok = bool(os.environ.get('NGROK_AUTHTOKEN'))
print(f"HF_TOKEN         : {'✅ set' if hf    else '⚠️  NOT SET — Gemma 4 download may fail'}")
print(f"NGROK_AUTHTOKEN  : {'✅ set' if ngrok else '⚠️  not set (Colab proxy used instead — OK)'}")

## 🚀 Cell 3 — Start Platform
_Installs inference deps, loads model, starts API + UI, creates HTTPS URL._

> ⏱ First run downloads Gemma 4 weights (~5 GB). Subsequent runs use cache.

In [ ]:
import sys, os

REPO_DIR = '/content/Duino_API'

# Re-register path (safe to repeat if kernel restarted)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

from studio.backend.colab import start

result = start(
    model    = 'gemma-4-2b',   # 'gemma-4-9b' or 'gemma-4-27b' for more VRAM
    api_port = 8000,
    ui_port  = 3000,
    expose   = True,           # creates public HTTPS URL via Colab proxy
)

api_url    = result['api_url']
ui_url     = result['ui_url']
api_key    = result['api_key']
embed_html = result['embed_html']

print(f'\n📡 API  : {api_url}')
print(f'🎨 UI   : {ui_url}')
print(f'🔑 Key  : {api_key}')
print(f'📖 Docs : {api_url}/docs')

## 🎨 Cell 4 — Embed React UI
_Interactive streaming chat UI inside this notebook._

In [ ]:
# Method A — Colab native port proxy (most reliable)
from google.colab import output as colab_output
colab_output.serve_kernel_port_as_iframe(3000, height=700, width='100%')

In [ ]:
# Method B — iframe with postMessage config (use if Method A doesn't show UI)
from IPython.display import HTML, display
display(HTML(embed_html))

## 🧪 Cell 5 — Test Inference API

In [ ]:
import requests

# Single request
resp = requests.post(
    f'{api_url}/v1/chat/completions',
    headers={'X-API-Key': api_key},
    json={
        'model': 'gemma-4-2b',
        'messages': [{'role': 'user', 'content': 'What is the capital of France? One sentence.'}],
        'max_tokens': 64,
    },
)
print('Status:', resp.status_code)
print('Answer:', resp.json()['choices'][0]['message']['content'])
print('Tokens:', resp.json()['usage'])

In [ ]:
# Streaming request
import requests, json

print('Streaming:')
with requests.post(
    f'{api_url}/v1/chat/completions',
    headers={'X-API-Key': api_key},
    json={'model': 'gemma-4-2b',
          'messages': [{'role': 'user', 'content': 'Write a short poem about AI.'}],
          'max_tokens': 128, 'stream': True},
    stream=True,
) as r:
    for line in r.iter_lines():
        if line:
            line = line.decode()
            if line.startswith('data: ') and line[6:] != '[DONE]':
                delta = json.loads(line[6:]).get('choices',[{}])[0].get('delta',{}).get('content','')
                print(delta, end='', flush=True)
print()

## 💬 Cell 6 — Multi-turn Session Chat

In [ ]:
import requests

sess = requests.post(f'{api_url}/v1/sessions',
    headers={'X-API-Key': api_key},
    json={'model_id': 'gemma-4-2b'}).json()
session_id = sess['session_id']
print(f'Session: {session_id}')

def chat(message):
    r = requests.post(f'{api_url}/v1/chat/completions',
        headers={'X-API-Key': api_key},
        json={'model': 'gemma-4-2b',
              'messages': [{'role': 'user', 'content': message}],
              'session_id': session_id, 'max_tokens': 256}).json()
    return r['choices'][0]['message']['content']

print('\nUser: My name is Jalal.')
print('AI  :', chat('My name is Jalal.'))
print('\nUser: What is my name?')
print('AI  :', chat('What is my name?'))  # Should remember

## ⚛️ Cell 7 — Run Any React Project Inside Colab
_Scaffold a Vite + React app, start the dev server, embed as iframe._

In [ ]:
import subprocess, threading, time, os
from google.colab import output as colab_output

REACT_PORT = 4000
REACT_DIR  = '/content/my-react-app'

if not os.path.exists(REACT_DIR):
    print('Scaffolding Vite + React...')
    !npm create vite@latest {REACT_DIR} -- --template react -y 2>/dev/null || \
     npm create vite@latest {REACT_DIR} -- --template react
    !npm install --prefix {REACT_DIR} --silent
    print('✅ React app ready')
else:
    print('✅ App exists')

def _vite():
    subprocess.run(['npm', 'run', 'dev', '--', '--host', '0.0.0.0',
                    '--port', str(REACT_PORT)], cwd=REACT_DIR)

threading.Thread(target=_vite, daemon=True).start()
time.sleep(6)
print(f'Vite running on port {REACT_PORT}')
colab_output.serve_kernel_port_as_iframe(REACT_PORT, height=600, width='100%')

In [ ]:
# Next.js inside Colab
import subprocess, threading, time, os
from google.colab import output as colab_output

NEXT_PORT = 4001
NEXT_DIR  = '/content/my-next-app'

if not os.path.exists(NEXT_DIR):
    !npx create-next-app@latest {NEXT_DIR} --ts --eslint --no-git --yes

def _next():
    subprocess.run(['npm', 'run', 'dev', '--', '-p', str(NEXT_PORT)], cwd=NEXT_DIR)

threading.Thread(target=_next, daemon=True).start()
time.sleep(8)
colab_output.serve_kernel_port_as_iframe(NEXT_PORT, height=600, width='100%')

## 📊 Cell 8 — Health & Models

In [ ]:
import requests

health = requests.get(f'{api_url}/v1/health').json()
models = requests.get(f'{api_url}/v1/models', headers={'X-API-Key': api_key}).json()

print('\n📊 Platform Health')
print('=' * 40)
for k, v in health.items():
    print(f'  {k:<22} : {v}')

print('\n🤖 Available Models')
print('=' * 40)
for m in models.get('data', []):
    print(f"  - {m['id']}")

## ♾️ Cell 9 — Keep Alive

In [ ]:
import time

print('Platform running — keeping session alive. Press STOP (■) to terminate.')
print(f'  API  : {api_url}')
print(f'  UI   : {ui_url}')
print(f'  Docs : {api_url}/docs')
print()

for i in range(10_000):
    time.sleep(300)
    print(f'= [{i+1} × 5min alive]', end='', flush=True)